# Reproduce preprocessing (optional)
Do not rerun just to train: use the supplied preprocessed data. Run 01_unpack_data first. All original assumptions and limitations are documented. This notebook uses the exact tested preprocessing functions.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [ ]:
"""Reproducible PathSnap candidate dataset; reference labels are derived proxies."""
from pathlib import Path
import argparse, hashlib, json, csv
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

FEATURES = ['distance_m','heading_cos','heading_available','segment_length_m',
            'projection_fraction','speed_mps','previous_gap_s','gps_step_m',
            'candidate_distance_rank','distance_minus_nearest_m']

def project_segments(points, starts, ends):
    delta = ends-starts
    fraction = np.clip(np.sum((points-starts)*delta,axis=-1)/np.maximum(np.sum(delta*delta,axis=-1),1e-12),0,1)
    distance = np.linalg.norm(points-(starts+fraction[...,None]*delta),axis=-1)
    return distance, fraction

def candidates(points, starts, ends, k=8):
    """Exact nearest segments: midpoint bound includes every segment within seed radius."""
    mids=(starts+ends)/2; half=np.linalg.norm(ends-starts,axis=1)/2
    tree=cKDTree(mids); count=min(k,len(starts))
    _, seeds=tree.query(points,k=count)
    seeds=np.asarray(seeds).reshape(len(points),count)
    result=[]; distances=[]; fractions=[]
    for p, seed in zip(points,seeds):
        ds,_=project_segments(p,starts[seed],ends[seed])
        ids=np.asarray(tree.query_ball_point(p,float(ds.max()+half.max()+1e-7)),dtype=int)
        d,f=project_segments(p,starts[ids],ends[ids]); order=np.lexsort((ids,d))[:count]
        result.append(ids[order]); distances.append(d[order]); fractions.append(f[order])
    return np.asarray(result),np.asarray(distances),np.asarray(fractions)

def align_reference(points, starts, ends, route):
    """Minimum sum squared distance with nondecreasing route occurrence indices.

    This is a label derivation heuristic, not verified observation-level truth.
    Free start/end and skipped occurrences support partially observed routes.
    """
    dist,_=project_segments(points[:,None,:],starts[route][None,:,:],ends[route][None,:,:])
    back=np.zeros(dist.shape,dtype=np.int32); costs=dist[0]**2
    for t in range(1,len(points)):
        # Prefix argmin, retaining earliest index on ties.
        prefix=np.minimum.accumulate(costs)
        indices=np.arange(len(route),dtype=np.int32)
        record=np.r_[True,costs[1:]<prefix[:-1]]
        arg=np.maximum.accumulate(np.where(record,indices,0))
        back[t]=arg; costs=dist[t]**2+prefix
    idx=np.empty(len(points),dtype=np.int32);idx[-1]=int(np.argmin(costs))
    for t in range(len(points)-1,0,-1): idx[t-1]=back[t,idx[t]]
    labels=route[idx]; residual=dist[np.arange(len(points)),idx]
    # Local geometric ambiguity among different reference arc IDs.
    other=np.where(route[None,:]!=labels[:,None],dist,np.inf).min(axis=1)
    margin=other-residual
    return labels,idx,residual,margin

def process(folder,out,fold):
    tid=folder.name
    files={ext:folder/f'{tid}.{ext}' for ext in ['track','nodes','arcs','route']}
    raw={ext:np.loadtxt(p,ndmin=2) for ext,p in files.items()}
    gps=raw['track']; nodes=raw['nodes']; arcs_raw=raw['arcs'];route_raw=raw['route'].ravel()
    assert gps.shape[1]==3 and nodes.shape[1]==2 and arcs_raw.shape[1]==2
    assert np.isfinite(nodes).all() and np.isfinite(arcs_raw).all() and np.isfinite(route_raw).all()
    assert np.equal(arcs_raw,np.floor(arcs_raw)).all() and np.equal(route_raw,np.floor(route_raw)).all()
    arcs=arcs_raw.astype(int);route=route_raw.astype(int)
    assert len(route)>0 and arcs.min()>=0 and arcs.max()<len(nodes) and route.min()>=0 and route.max()<len(arcs)
    valid=np.isfinite(gps).all(axis=1)&(np.abs(gps[:,0])<=180)&(np.abs(gps[:,1])<=90)
    original=np.flatnonzero(valid);gps=gps[valid]
    order=np.argsort(gps[:,2],kind='stable');gps=gps[order];original=original[order]
    # Keep same-time observations; undefined speeds/headings receive zero plus availability flag.
    lon,lat=np.median(nodes,axis=0)
    def project(coords):
        radians=np.deg2rad(coords-np.array([lon,lat]))
        return 6371008.8*radians*np.array([np.cos(np.deg2rad(lat)),1.])
    xy=project(gps[:,:2]);nxy=project(nodes)
    starts=nxy[arcs[:,0]];ends=nxy[arcs[:,1]];delta=ends-starts;length=np.linalg.norm(delta,axis=1)
    ids,distance,fraction=candidates(xy,starts,ends)
    labels,occ,residual,margin=align_reference(xy,starts,ends,route)
    dt=np.r_[0,np.diff(gps[:,2])];movement=np.vstack([np.zeros(2),np.diff(xy,axis=0)]);step=np.linalg.norm(movement,axis=1)
    usable=(dt>0)&(dt<=30)&(step>=2)
    speed=np.divide(step,dt,out=np.zeros_like(step),where=dt>0)
    cos=np.sum(movement[:,None,:]*delta[ids],axis=-1)/np.maximum(step[:,None]*length[ids],1e-12)
    cos=np.where(usable[:,None],cos,0)
    shape=ids.shape;repeat=lambda x:np.broadcast_to(x[:,None],shape)
    X=np.stack([distance,cos,repeat(usable.astype(float)),length[ids],fraction,repeat(speed),repeat(dt),repeat(step),np.broadcast_to(np.arange(shape[1]),shape),distance-distance[:,:1]],axis=-1).astype(np.float32)
    y=(ids==labels[:,None]).astype(np.int8)
    reliable=(residual<=50)&(margin>=3)
    train_mask=reliable & y.any(axis=1)
    destination=out/folder.parent.name;destination.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(destination/f'{tid}.npz',X=X,y=y,candidate_arc_ids=ids,reference_arc_id=labels,reference_occurrence=occ,reference_distance_m=residual,reference_margin_m=margin,label_reliable=reliable,training_observation_mask=train_mask,original_row=original,gps=gps,xy_m=xy,fold=np.int16(fold),projection_origin_lonlat=np.array([lon,lat]))
    graph=coo_matrix((np.ones(len(arcs)),(arcs[:,0],arcs[:,1])),shape=(len(nodes),len(nodes))).tocsr()
    components,_=connected_components(graph,directed=True,connection='weak')
    present=y.any(axis=1)
    return dict(split=folder.parent.name,trip_id=tid,fold=fold,raw_observations=len(raw['track']),observations=len(gps),removed_invalid=int((~valid).sum()),reordered_rows=int(np.sum(order!=np.arange(len(order)))),duplicate_timestamps=int(np.sum(dt[1:]==0)),gaps_over_30s=int(np.sum(dt>30)),max_gap_s=float(dt.max()),nodes=len(nodes),arcs=len(arcs),zero_length_arcs=int(np.sum(length==0)),weak_components=components,isolated_nodes=int(np.sum(np.bincount(arcs.ravel(),minlength=len(nodes))==0)),route_segments=len(route),route_connected_fraction=float(np.mean(arcs[route[:-1],1]==arcs[route[1:],0])) if len(route)>1 else 1.,candidate_misses=int((~present).sum()),reliable_labels=int(reliable.sum()),training_observations=int(train_mask.sum()),reference_distance_median=float(np.median(residual)),reference_distance_p95=float(np.percentile(residual,95)),projection_latitude=float(lat),lon_min=float(gps[:,0].min()),lon_max=float(gps[:,0].max()),lat_min=float(gps[:,1].min()),lat_max=float(gps[:,1].max()))

def main():
    p=argparse.ArgumentParser();p.add_argument('--raw',type=Path,default=Path('Student folder'));p.add_argument('--out',type=Path,default=Path('processed'));args=p.parse_args()
    args.out.mkdir(exist_ok=True,parents=True)
    train=sorted((args.raw/'train').iterdir());rng=np.random.default_rng(212210209208); perm=rng.permutation(len(train));folds={train[int(i)].name:int(j%5) for j,i in enumerate(perm)}
    rows=[];manifest={}
    for split in ['train','student_test']:
        for folder in sorted((args.raw/split).iterdir()):
            if not folder.is_dir():continue
            row=process(folder,args.out,folds.get(folder.name,-1));rows.append(row)
            for f in sorted(folder.iterdir()):
                if f.is_file():manifest[str(f.relative_to(args.raw))]=hashlib.sha256(f.read_bytes()).hexdigest()
            print(split,folder.name,'observations',row['observations'],'usable',row['training_observations'],flush=True)
    with (args.out/'inventory.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    (args.out/'schema.json').write_text(json.dumps(dict(features=FEATURES,label_kind='derived_reference_proxy',k=8,alignment='monotone route distance minimisation',reliable_max_distance_m=50,reliable_min_margin_m=3,seed=212210209208,folds=5),indent=2))
    (args.out/'raw_sha256.json').write_text(json.dumps(manifest,indent=2))
    print('Finished',len(rows),'trips',flush=True)



In [ ]:
import sys
_saved_argv = sys.argv
try:
    sys.argv = ["preprocess"]
    main()
finally:
    sys.argv = _saved_argv

## Validate full data and generate a development map
Reference layers are development-only, never the evaluator UI.

In [ ]:
"""Validate generated artifacts and produce a development Folium map."""
import csv,json,hashlib
from pathlib import Path
import numpy as np
import folium

root=Path('processed');reports=Path('reports');reports.mkdir(exist_ok=True)
summary={};failures=[];fold_counts={};max_distortion=0.
for split,expected in [('train',80),('student_test',10)]:
    files=sorted((root/split).glob('*.npz'));assert len(files)==expected
    stats=dict(trips=len(files),observations=0,candidate_rows=0,training_observations=0,reliable_labels=0,candidate_misses=0)
    for p in files:
        with np.load(p,allow_pickle=False) as d:
            X=d['X'];y=d['y'];n=len(X)
            assert X.shape==(n,8,len(FEATURES)) and y.shape==(n,8)
            assert np.isfinite(X).all() and np.all(np.diff(d['gps'][:,2])>=0)
            assert np.array_equal(y,d['candidate_arc_ids']==d['reference_arc_id'][:,None])
            assert np.all(y.sum(axis=1)<=1)
            assert np.all(np.diff(d['reference_occurrence'])>=0)
            assert np.array_equal(d['training_observation_mask'],d['label_reliable']&y.any(axis=1))
            assert np.all(np.diff(X[:,:,0],axis=1)>=-1e-5)
            assert len(np.unique(d['original_row']))==n
            fold=int(d['fold']);assert fold in (range(5) if split=='train' else [-1])
            if split=='train':fold_counts[fold]=fold_counts.get(fold,0)+1
            stats['observations']+=n;stats['candidate_rows']+=n*8;stats['training_observations']+=int(d['training_observation_mask'].sum());stats['reliable_labels']+=int(d['label_reliable'].sum());stats['candidate_misses']+=int((~y.any(axis=1)).sum())
        folder=Path('Student folder')/split/p.stem
        nodes=np.loadtxt(folder/f'{p.stem}.nodes');arcs=np.loadtxt(folder/f'{p.stem}.arcs',dtype=int)
        with np.load(p) as d:lon,lat=d['projection_origin_lonlat']
        rad=np.deg2rad(nodes);a=rad[arcs[:,0]];b=rad[arcs[:,1]];diff=b-a
        h=np.sin(diff[:,1]/2)**2+np.cos(a[:,1])*np.cos(b[:,1])*np.sin(diff[:,0]/2)**2
        geodesic=2*6371008.8*np.arcsin(np.sqrt(np.clip(h,0,1)))
        xy=6371008.8*np.deg2rad(nodes-[lon,lat])*[np.cos(np.deg2rad(lat)),1]
        planar=np.linalg.norm(xy[arcs[:,1]]-xy[arcs[:,0]],axis=1)
        good=geodesic>1
        max_distortion=max(max_distortion,float(np.max(np.abs(planar[good]/geodesic[good]-1))))
    stats['candidate_miss_rate']=stats['candidate_misses']/stats['observations']
    stats['fitting_coverage']=stats['training_observations']/stats['observations'];summary[split]=stats
assert fold_counts=={i:16 for i in range(5)}
summary['fold_trip_counts']=fold_counts;summary['max_segment_distance_distortion_vs_haversine']=max_distortion
# Independently check exact candidates against all road segments for ten real GPS points.
p=sorted((root/'train').glob('*.npz'))[0];folder=Path('Student folder/train')/p.stem
nodes=np.loadtxt(folder/f'{p.stem}.nodes');arcs=np.loadtxt(folder/f'{p.stem}.arcs',dtype=int);route=np.loadtxt(folder/f'{p.stem}.route',dtype=int)
with np.load(p) as d:
    lon,lat=d['projection_origin_lonlat'];xy=6371008.8*np.deg2rad(nodes-[lon,lat])*[np.cos(np.deg2rad(lat)),1]
    points=d['xy_m'][np.linspace(0,len(d['X'])-1,10,dtype=int)]
    ids,dist,_=candidates(points,xy[arcs[:,0]],xy[arcs[:,1]])
    brute,_=project_segments(points[:,None,:],xy[arcs[:,0]][None,:,:],xy[arcs[:,1]][None,:,:])
    np.testing.assert_allclose(dist,np.sort(brute,axis=1)[:,:8],atol=1e-7)
    summary['real_candidate_checks']=10
    # Map is development-only: reference overlay must never enter evaluator UI.
    gps=d['gps'];m=folium.Map(location=[float(gps[:,1].mean()),float(gps[:,0].mean())],zoom_start=13)
    roads=folium.FeatureGroup(name='Road network (near observed trip)')
    selected=np.unique(d['candidate_arc_ids'])
    for arc in selected:
        folium.PolyLine(nodes[arcs[arc]][:,[1,0]].tolist(),color='gray',weight=1,opacity=.4).add_to(roads)
    roads.add_to(m)
    ref=folium.FeatureGroup(name='Supplied reference traversal')
    for arc in route:folium.PolyLine(nodes[arcs[arc]][:,[1,0]].tolist(),color='green',weight=4).add_to(ref)
    ref.add_to(m)
    raw=folium.FeatureGroup(name='Raw GPS observations')
    for i,row in enumerate(gps):folium.CircleMarker([float(row[1]),float(row[0])],radius=2,color='blue',tooltip=f'GPS {i}').add_to(raw)
    raw.add_to(m)
    baseline=folium.FeatureGroup(name='Nearest-road baseline (untrained, disconnected segments possible)')
    for arc in np.unique(d['candidate_arc_ids'][:,0]):folium.PolyLine(nodes[arcs[arc]][:,[1,0]].tolist(),color='orange',weight=2).add_to(baseline)
    baseline.add_to(m);folium.LayerControl().add_to(m);m.fit_bounds([[gps[:,1].min(),gps[:,0].min()],[gps[:,1].max(),gps[:,0].max()]])
    m.save(str(reports/f'development_trip_{p.stem}.html'))
# Recheck provenance of every original input.
manifest=json.loads((root/'raw_sha256.json').read_text())
for name,digest in manifest.items():assert hashlib.sha256((Path('Student folder')/name).read_bytes()).hexdigest()==digest
summary['raw_files_hash_verified']=len(manifest);summary['all_checks_passed']=True
(reports/'validation_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
